# CSI Presence Detection Pipeline
Thu thập CSI từ ESP32, extract features, train classifier phát hiện có người / không người.

In [ ]:
import os
import csv
import gc
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.signal import butter, sosfiltfilt, find_peaks
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import plotly.graph_objects as go
import plotly.figure_factory as ff

print('Import OK')

# Config

In [ ]:
# ── Đường dẫn & label ────────────────────────────────────────────────────
DATA_ROOT = 'WiVSD/Router'   # ← đổi thành path chứa các thư mục class

LABEL_MAP = {
    '1 nguoi nam bat quat' : 'co_nguoi',
    '1 nguoi ngoi'         : 'co_nguoi',
    '1 nguoi ngoi bat quat': 'co_nguoi',
    '2 nguoi ngoi bat quat': 'co_nguoi',
    'nhieu nguoi bat quat' : 'co_nguoi',
    'khong nguoi'          : 'khong_nguoi',
    'khong nguoi bat quat' : 'khong_nguoi',
}

# ── Signal processing ────────────────────────────────────────────────────
FS_ASSUMED     = 100.0   # Hz — dùng khi không có timestamp
FS_TARGET      = 60.0    # Hz — resample về
GAUSSIAN_SIGMA = 15      # smoothing
BUTTER_ORDER   = 2
BUTTER_CUTOFF  = 1.0     # lowpass Hz
MIN_PEAK_DIST  = int(FS_TARGET * 60 / 30)   # min 2s giữa peaks (~30 bpm max)

# Subcarrier pilot/DC cần bỏ
_DROP = {
    64 : list(range(0, 6))  + list(range(59, 64))  + [32],
    128: list(range(0, 6))  + list(range(122, 128)) + [63, 64, 65],
    256: list(range(0, 11)) + list(range(245, 256)) + [127, 128, 129],
}

print('Config OK')

# Helper Functions

In [ ]:
def load_file_lowram(fpath):
    """
    Đọc CSV từng dòng — không load toàn bộ vào RAM.
    Trả về: (sig_best_sub, timestamps)
    """
    rows_amp = []
    ts_list  = []
    n_sub_global = None

    with open(fpath, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Parse I/Q → amplitude (tất cả subcarrier)
            s   = row.get('data', '')
            if not s or s == 'nan':
                continue
            arr = np.fromstring(s[1:-1], dtype=np.float32, sep=',')
            if len(arr) == 0:
                continue
            i, q = arr[0::2], arr[1::2]
            n_sub = len(i)
            n_sub_global = n_sub

            # Bỏ pilot/DC
            drop = _DROP.get(n_sub * 2)  # len = n_sub*2
            if drop:
                mask = np.ones(n_sub, dtype=bool)
                valid_drop = [d for d in drop if d < n_sub]
                mask[valid_drop] = False
                i, q = i[mask], q[mask]

            rows_amp.append(np.hypot(i, q))

            # Timestamp
            ts_val = None
            for col in ['timestamp', 'local_timestamp']:
                v = row.get(col, '')
                if v and v not in ('', 'nan', 'None'):
                    try:
                        ts_val = float(v)
                        break
                    except:
                        pass
            ts_list.append(ts_val if ts_val is not None else np.nan)

    if len(rows_amp) == 0:
        raise ValueError('File rỗng hoặc không parse được')

    amp_mat = np.stack(rows_amp, axis=0).astype(np.float32)  # (n_rows, n_sub)
    del rows_amp

    # Timestamps
    ts = np.array(ts_list, dtype=np.float64)
    nan_ratio = np.isnan(ts).mean()
    if nan_ratio > 0.1:
        ts = np.arange(len(ts)) / FS_ASSUMED
    else:
        # Interpolate NaN
        nans = np.isnan(ts)
        ts[nans] = np.interp(np.where(nans)[0], np.where(~nans)[0], ts[~nans])

    # Tìm best subcarrier theo variance
    best_sub = int(np.argmax(amp_mat.var(axis=0)))
    sig = amp_mat[:, best_sub].astype(np.float64)
    del amp_mat
    gc.collect()

    return sig, ts


def _hampel_1d(arr, window_size=20, n_sigma=3.0):
    """Vectorized Hampel filter."""
    s = pd.Series(arr.astype(float))
    w = 2 * window_size + 1
    rolling_med = s.rolling(window=w, center=True, min_periods=1).median()
    rolling_mad = (s - rolling_med).abs().rolling(window=w, center=True, min_periods=1).median()
    threshold   = n_sigma * 1.4826 * rolling_mad
    outliers    = np.abs(arr - rolling_med.values) > threshold.values
    result      = arr.copy()
    result[outliers] = rolling_med.values[outliers]
    return result


def process_signal(sig, ts_raw):
    """
    Hampel → Gaussian → resample → Butterworth lowpass → detrend
    """
    # 1. Hampel
    sig = _hampel_1d(sig)

    # 2. Gaussian smooth
    sig = gaussian_filter1d(sig, sigma=GAUSSIAN_SIGMA)

    # 3. Resample về FS_TARGET
    # Đảm bảo ts_raw tăng đơn điệu
    ts = ts_raw.copy()
    dt = np.diff(ts)
    bad = dt <= 0
    if bad.any():
        # Fix timestamp không đơn điệu
        ts = np.linspace(ts[0], ts[0] + len(ts)/FS_ASSUMED, len(ts))

    t_uniform = np.arange(ts[0], ts[-1], 1.0 / FS_TARGET)
    if len(t_uniform) < 10:
        raise ValueError('Signal quá ngắn sau resample')

    sig = interp1d(ts, sig, kind='linear',
                   bounds_error=False,
                   fill_value=(sig[0], sig[-1]))(t_uniform)

    # 4. Butterworth lowpass
    sos = butter(BUTTER_ORDER, BUTTER_CUTOFF, btype='low', fs=FS_TARGET, output='sos')
    sig = sosfiltfilt(sos, sig)

    # 5. Detrend tuyến tính
    x   = np.arange(len(sig))
    sig = sig - np.polyval(np.polyfit(x, sig, 1), x)

    return sig, t_uniform


def extract_features(sig):
    """Trích 13 features từ signal đã processed."""
    peaks,   _ = find_peaks(sig,  distance=MIN_PEAK_DIST)
    troughs, _ = find_peaks(-sig, distance=MIN_PEAK_DIST)

    # Peak periodicity
    if len(peaks) >= 2:
        intervals  = np.diff(peaks) / FS_TARGET
        mean_period = intervals.mean()
        std_period  = intervals.std()
        dom_freq    = 1.0 / mean_period if mean_period > 0 else 0
    else:
        mean_period = std_period = dom_freq = 0.0

    # Amplitude
    if len(peaks) > 0 and len(troughs) > 0:
        amp_mean = (sig[peaks].mean() - sig[troughs].mean()) / 2
        amp_std  = sig[peaks].std()
    else:
        amp_mean = amp_std = 0.0

    # Spectral energy
    fft_mag  = np.abs(np.fft.rfft(sig))
    fft_freq = np.fft.rfftfreq(len(sig), d=1.0 / FS_TARGET)
    b_mask   = (fft_freq >= 0.1) & (fft_freq <= 0.5)
    n_mask   = (fft_freq >  0.5) & (fft_freq <= 2.0)
    e_breath = fft_mag[b_mask].mean() if b_mask.any() else 0.0
    e_noise  = fft_mag[n_mask].mean() if n_mask.any() else 1e-10
    snr      = e_breath / (e_noise + 1e-10)

    return {
        'variance'     : float(sig.var()),
        'std'          : float(sig.std()),
        'peak_count'   : len(peaks),
        'dom_freq_hz'  : float(dom_freq),
        'mean_period'  : float(mean_period),
        'std_period'   : float(std_period),
        'amp_mean'     : float(amp_mean),
        'amp_std'      : float(amp_std),
        'energy_breath': float(e_breath),
        'energy_noise' : float(e_noise),
        'snr'          : float(snr),
        'kurtosis'     : float(pd.Series(sig).kurtosis()),
        'skewness'     : float(pd.Series(sig).skew()),
    }


print('Helper functions OK')

# Load Dataset

In [ ]:
records = []

for class_dir in sorted(os.listdir(DATA_ROOT)):
    class_path = os.path.join(DATA_ROOT, class_dir)
    if not os.path.isdir(class_path):
        continue

    label = LABEL_MAP.get(class_dir)
    if label is None:
        print(f'  [skip] {class_dir}')
        continue

    csv_files = sorted([f for f in os.listdir(class_path) if f.endswith('.csv')])
    print(f'  {class_dir} → {label}: {len(csv_files)} files')

    for fname in csv_files:
        fpath = os.path.join(class_path, fname)
        try:
            sig_raw, ts = load_file_lowram(fpath)
            sig_proc, _ = process_signal(sig_raw, ts)
            del sig_raw, ts

            feats = extract_features(sig_proc)
            del sig_proc
            gc.collect()

            feats.update({'label': label, 'class': class_dir, 'file': fname})
            records.append(feats)

        except Exception as e:
            print(f'    [lỗi] {fname}: {e}')

df_feat = pd.DataFrame(records)
print(f'\nTổng: {len(df_feat)} files')
print(df_feat['label'].value_counts())
df_feat.head()

# Train & Evaluate

In [ ]:
FEATURE_COLS = [
    'variance', 'std', 'peak_count', 'dom_freq_hz', 'mean_period',
    'std_period', 'amp_mean', 'amp_std', 'energy_breath',
    'energy_noise', 'snr', 'kurtosis', 'skewness'
]

X  = df_feat[FEATURE_COLS].fillna(0).values
le = LabelEncoder().fit(df_feat['label'])
y  = le.transform(df_feat['label'])
label_names = le.classes_

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5-fold cross-validation
clf    = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring='accuracy')

print(f'CV Accuracy : {scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%')
print(f'Per-fold    : {[f"{s*100:.1f}%" for s in scores]}')

# Train full
clf.fit(X_scaled, y)
y_pred = clf.predict(X_scaled)
print('\n' + classification_report(y, y_pred, target_names=label_names))

# Confusion Matrix & Feature Importance

In [ ]:
# Confusion matrix
cm  = confusion_matrix(y, y_pred)
fig = ff.create_annotated_heatmap(
    cm, x=list(label_names), y=list(label_names),
    colorscale='Blues', showscale=True
)
fig.update_layout(
    title='Confusion Matrix (train set)',
    xaxis_title='Predicted', yaxis_title='Actual', height=400
)
fig.show()

# Feature importance
imp = pd.Series(clf.feature_importances_, index=FEATURE_COLS).sort_values()
fig2 = go.Figure(go.Bar(x=imp.values, y=imp.index, orientation='h',
                         marker_color='steelblue'))
fig2.update_layout(title='Feature Importance', height=450,
                   xaxis_title='Importance', yaxis_title='Feature')
fig2.show()

# Predict 1 file mới

In [ ]:
def predict_file(fpath):
    """Predict label cho 1 file CSV mới."""
    sig_raw, ts = load_file_lowram(fpath)
    sig_proc, t = process_signal(sig_raw, ts)

    # Plot signal
    fig = go.Figure(go.Scatter(
        x=t - t[0], y=sig_proc,
        mode='lines', line=dict(width=1.5)
    ))
    fig.update_layout(
        title=os.path.basename(fpath),
        xaxis_title='Thời gian (s)',
        yaxis_title='Amplitude (detrended)', height=320
    )
    fig.show()

    feats  = extract_features(sig_proc)
    X_new  = scaler.transform([list(feats[c] for c in FEATURE_COLS)])
    pred   = clf.predict(X_new)[0]
    proba  = clf.predict_proba(X_new)[0]

    print(f'Kết quả  : {le.inverse_transform([pred])[0]}')
    for name, p in zip(label_names, proba):
        print(f'  {name:20s}: {p*100:.1f}%')

# Ví dụ:
# predict_file('WiVSD/Router/1 nguoi ngoi/test.csv')